# Week 5, Lab 2 — AutoGen GroupChat


In [1]:
WEEK = 'Week 5'
LAB = 'Lab 2 — GroupChat'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 5 / Lab 2 — GroupChat
Backend: ollama
Need Ollama running: `ollama serve` and `ollama pull llama3.2:1b`
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [2]:
cfg = openai_client_kwargs()
llm_config = {"config_list": [{
    "model": cfg["model"], "base_url": cfg["base_url"], "api_key": cfg["api_key"], "price": [0, 0],
}], "temperature": 0.2}

import autogen

researcher = autogen.AssistantAgent("researcher", llm_config=llm_config, system_message="Give 3 factual bullets. Then sit back.")
writer = autogen.AssistantAgent("writer", llm_config=llm_config, system_message="Turn bullets into 4 student sentences.")
critic = autogen.AssistantAgent("critic", llm_config=llm_config, system_message="One critique sentence, then say TERMINATE if good enough.")
user = autogen.UserProxyAgent("user", human_input_mode="NEVER", code_execution_config=False, max_consecutive_auto_reply=1)

chat = autogen.GroupChat(agents=[user, researcher, writer, critic], messages=[], max_round=6)
manager = autogen.GroupChatManager(groupchat=chat, llm_config=llm_config)
user.initiate_chat(manager, message="Explain Ollama to a student who has only used Colab.")


user (to chat_manager):

Explain Ollama to a student who has only used Colab.

--------------------------------------------------------------------------------

Next speaker: researcher

researcher (to chat_manager):

Sure, I can explain Ollama in simple terms for you! Here are three key points:

1. **Ollama is a new AI language model**: Just like Google's Colab uses the powerful BERT model, Ollama is another advanced artificial intelligence system designed to understand and generate human-like text.

2. **Training Data Difference**: While BERT focuses on understanding context in sentences (like identifying synonyms or related words), Ollama has been trained on a broader range of data that includes more diverse topics and styles of writing, making it potentially better at generating coherent and varied responses across different contexts.

3. **Potential for Creativity and Innovation**: One exciting aspect of using AI like Ollama is its potential to assist in creative tasks such as sto

ChatResult(chat_id=208003161656507445235773431426544686893, chat_history=[{'content': 'Explain Ollama to a student who has only used Colab.', 'role': 'assistant', 'name': 'user'}, {'content': "Sure, I can explain Ollama in simple terms for you! Here are three key points:\n\n1. **Ollama is a new AI language model**: Just like Google's Colab uses the powerful BERT model, Ollama is another advanced artificial intelligence system designed to understand and generate human-like text.\n\n2. **Training Data Difference**: While BERT focuses on understanding context in sentences (like identifying synonyms or related words), Ollama has been trained on a broader range of data that includes more diverse topics and styles of writing, making it potentially better at generating coherent and varied responses across different contexts.\n\n3. **Potential for Creativity and Innovation**: One exciting aspect of using AI like Ollama is its potential to assist in creative tasks such as story generation, scri

Keep `max_round` tiny on small models.
